<a href="https://colab.research.google.com/github/Gabnyb/Alzheimer-s-Analytics/blob/main/INFO381A_LR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub as kh
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, accuracy_score, f1_score

Load dataset

In [ ]:
path = kh.dataset_download("rabieelkharoua/alzheimers-disease-dataset")
print("Path to dataset files:", path)

files = os.listdir(path)
print("Files in directory:", files)

csv_file = [f for f in files if f.endswith('.csv')][0]
csv_path = os.path.join(path, csv_file)


df = pd.read_csv(csv_path)
print("Shape:", df.shape)
df.head()

Basic overview

In [ ]:
df.info()
df.describe()

Check for missing values

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]
missing.sort_values(ascending=False)

Class distribution

In [ ]:
df['Diagnosis'].value_counts()
df['Diagnosis'].value_counts(normalize=True)

In [ ]:
sns.countplot(x='Diagnosis', data=df)
plt.title("Class Distribution")
plt.show()

Drop irrelevant columns

In [ ]:
df = df.drop(columns=['PatientID', 'DoctorInCharge'], errors='ignore')

Separate feature types

In [ ]:
binary_cols = [
    'Gender', 'Smoking', 'FamilyHistoryAlzheimers',
    'CardiovascularDisease', 'Diabetes', 'Depression',
    'HeadInjury', 'Hypertension',
    'MemoryComplaints', 'BehavioralProblems',
    'Confusion', 'Disorientation',
    'PersonalityChanges', 'DifficultyCompletingTasks',
    'Forgetfulness'
]

categorical_cols = ['Ethnicity', 'EducationLevel']

target_col = 'Diagnosis'

numeric_cols = [col for col in df.columns
                if col not in binary_cols + categorical_cols + [target_col]]

Verify

In [ ]:
print("Numeric:", numeric_cols)
print("Binary:", binary_cols)
print("Categorical:", categorical_cols)

Correlation Matrix

In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(df[numeric_cols].corr(), cmap='magma', center=0)
plt.title("Correlation Matrix (Numerical Features)")
plt.show()

FEATURE VS TARGET

Age vs. Diagnosis

In [ ]:
sns.boxplot(x='Diagnosis', y='Age', data=df)
plt.title("Age Distribution by Diagnosis")
plt.show()

MMSE vs Diagnosis

In [ ]:
sns.boxplot(x='Diagnosis', y='MMSE', data=df)
plt.title("MMSE Distribution by Diagnosis")
plt.show()

Quick statistical comparison

In [ ]:
df.groupby('Diagnosis')[numeric_cols].mean()

Prepare Data for Modeling

In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col]

# One-hot encode categorical variables
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print("Final feature shape:", X.shape)

Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Scale Features

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Logistic Regression Baseline

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

log_reg = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)
y_prob_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression Results")
print(classification_report(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))

Significance table

In [ ]:
import statsmodels.api as sm
import pandas as pd
import numpy as np

# 1. Prepare data (ensure constant is added)
X_train_sm = sm.add_constant(X_train_scaled)
feature_names = ['(Intercept)'] + list(X_train.columns)

# 2. Fit the model
model_sm = sm.Logit(y_train.values, X_train_sm)
result = model_sm.fit(disp=0)

# 3. Extract Global Model Fit Metrics
pseudo_r2 = result.prsquared
ll_ratio_p = result.llr_pvalue  # Likelihood Ratio Test p-value

print(f"--- Model Fit Statistics ---")
print(f"McFadden's Pseudo-R2: {pseudo_r2:.4f}")
print(f"LRT P-Value: {ll_ratio_p:.4e} (Model is significant if < 0.05)")
print("-" * 30)

# 4. Create the detailed Odds Ratio Table with 95% CI
params = result.params
conf = result.conf_int()
p_values = result.pvalues

# Build DataFrame
or_table = pd.DataFrame({
    'Coefficient (B)': params,
    'Std. Error': result.bse,
    'P-Value': p_values,
    'Odds Ratio (OR)': np.exp(params),
    '95% CI Lower': np.exp(conf[:, 0]),
    '95% CI Upper': np.exp(conf[:, 1])
})

# 5. Add Significance Stars for easier reading
def sig_stars(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else: return 'ns'

or_table['Sig.'] = or_table['P-Value'].apply(sig_stars)
or_table.index = feature_names

# Reorder columns for the report
final_cols = ['Coefficient (B)', 'P-Value', 'Sig.', 'Odds Ratio (OR)', '95% CI Lower', '95% CI Upper']
display(or_table[final_cols].round(4))

Forest Plot visual

In [ ]:
import matplotlib.pyplot as plt

# Exclude intercept for better scaling of the plot
plot_table = or_table.drop('(Intercept)')

plt.figure(figsize=(10, 8))
plt.errorbar(plot_table['Odds Ratio (OR)'], plot_table.index,
             xerr=[plot_table['Odds Ratio (OR)'] - plot_table['95% CI Lower'],
                   plot_table['95% CI Upper'] - plot_table['Odds Ratio (OR)']],
             fmt='o', color='black', capsize=5)

plt.axvline(x=1, color='red', linestyle='--') # Red line at OR=1 (No effect)
plt.xlabel('Odds Ratio (95% CI)')
plt.title('Clinical Risk Factors: Impact on Alzheimer\'s Diagnosis')
plt.grid(axis='x', alpha=0.3)
plt.show()

ROC Curve and confusion matrix

In [ ]:
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay, roc_curve, auc

# --- NEW: Save the data for the Comparison Plot ---
# This "captures" the numbers before the plot is even drawn
fpr_full, tpr_full, _ = roc_curve(y_test, y_prob_lr)
roc_auc_full = auc(fpr_full, tpr_full)

# --- Your Original Plotting Code ---
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
RocCurveDisplay.from_predictions(y_test, y_prob_lr, ax=ax[0], color='darkorange')
ax[0].plot([0, 1], [0, 1], color='navy', linestyle='--')
ax[0].set_title(f'Baseline ROC Curve (AUC = {roc_auc_full:.2f})')

# Confusion Matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, cmap='Blues', ax=ax[1])
ax[1].set_title('Baseline Confusion Matrix')

plt.tight_layout()
plt.show()

5-fold cross-validation

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# 1. Set up the cross-validation strategy
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2. Calculate ROC-AUC across 5 folds
cv_results = cross_val_score(log_reg, X_train_scaled, y_train,
                             cv=skf, scoring='roc_auc')

print(f"Baseline CV ROC-AUC Scores: {cv_results}")
print(f"Mean ROC-AUC: {cv_results.mean():.4f} (+/- {cv_results.std() * 2:.4f})")

CV Performance Boxplot

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=cv_results, color='lightblue')
sns.swarmplot(x=cv_results, color='black', alpha=0.7) # Shows the individual fold points
plt.title('5-Fold Cross-Validation: Baseline ROC-AUC')
plt.xlabel('ROC-AUC Score')
plt.show()

Combined Reduced-Feature Analysis


In [ ]:
import statsmodels.api as sm
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, accuracy_score, f1_score

# --- 1. DATA SUBSETTING (The "Reduced" Experiment) ---
# We exclude direct cognitive markers to see if lifestyle/demographics alone work
exclude_cols = [
    'MMSE', 'ADL', 'FunctionalAssessment',
    'MemoryComplaints', 'BehavioralProblems',
    'Confusion', 'Disorientation', 'PersonalityChanges',
    'DifficultyCompletingTasks', 'Forgetfulness'
]

X_reduced = X.drop(columns=exclude_cols, errors='ignore')

# --- 2. TRAIN/TEST SPLIT & SCALING ---
X_train_red, X_test_red, y_train_red, y_test_red = train_test_split(
    X_reduced, y, test_size=0.2, stratify=y, random_state=42
)

scaler_red = StandardScaler()
X_train_red_scaled = scaler_red.fit_transform(X_train_red)
X_test_red_scaled = scaler_red.transform(X_test_red)

# --- 3. SKLEARN MODEL (For ML Metrics) ---
log_reg_red = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
log_reg_red.fit(X_train_red_scaled, y_train_red)

y_prob_red = log_reg_red.predict_proba(X_test_red_scaled)[:, 1]
y_pred_red = log_reg_red.predict(X_test_red_scaled)
print(f"Reduced-Feature ROC-AUC: {roc_auc_score(y_test_red, y_prob_red):.4f}")

# --- 4. STATSMODELS (Fixed for Pandas Indexing) ---
X_train_red_sm = sm.add_constant(X_train_red_scaled)
y_train_numeric = y_train_red.astype(float)

model_red_sm = sm.Logit(y_train_numeric, X_train_red_sm)
res_red = model_red_sm.fit(disp=0)

# Extract Model Fit
print(f"Reduced Model McFadden's Pseudo-R2: {res_red.prsquared:.4f}")
print("-" * 30)

# Build the Detailed Table using .iloc for the Confidence Intervals
params_red = res_red.params
conf_red = res_red.conf_int() # This is a DataFrame
p_vals_red = res_red.pvalues
acc_red = accuracy_score(y_test_red, y_pred_red)
f1_red = f1_score(y_test_red, y_pred_red)
auc_red = roc_auc_score(y_test_red, y_prob_red)

red_summary_table = pd.DataFrame({
    'Coefficient (B)': params_red,
    'P-Value': p_vals_red,
    'Odds Ratio (OR)': np.exp(params_red),
    '95% CI Lower': np.exp(conf_red.iloc[:, 0]), # Use .iloc for the first column
    '95% CI Upper': np.exp(conf_red.iloc[:, 1])  # Use .iloc for the second column
})
print("--- REDUCED FEATURE RESULTS (Lifestyle Only) ---")
print(f"Accuracy: {acc_red:.4f}")
print(f"F1-Score: {f1_red:.4f}")
print("\nReduced Classification Report:")
print(classification_report(y_test_red, y_pred_red))

In [ ]:
# 1. Use the table name from the previous cell
# For the Reduced model, use 'red_summary_table'
plot_df = red_summary_table.iloc[1:].copy().reset_index().rename(columns={'index': 'Feature'})

# 2. Sort by Coefficient for a cleaner visual "ladder" effect
plot_df = plot_df.sort_values('Coefficient (B)')

plt.figure(figsize=(10, 10))

# 3. Use 1.96 * Standard Error for the 95% Confidence Interval
# We need to grab Std. Error from our summary (if you didn't include it in the table,
# we can calculate it or use the CI bounds we already have)
plt.errorbar(plot_df['Coefficient (B)'],
             plot_df['Feature'],
             xerr=[plot_df['Coefficient (B)'] - np.log(plot_df['95% CI Lower']),
                   np.log(plot_df['95% CI Upper']) - plot_df['Coefficient (B)']],
             fmt='o', color='black', capsize=3, elinewidth=1)

# 4. Vertical line at 0 (In Log-Odds, 0 means "No Effect")
plt.axvline(0, color='red', linestyle='--', linewidth=1)

plt.title('Baseline Feature Impact (Standardized Coefficients)', fontsize=14)
plt.xlabel('Log-Odds (Direction and Magnitude of Impact)', fontsize=12)
plt.grid(axis='x', linestyle=':', alpha=0.7)

plt.tight_layout()
plt.show()

Logistic Regression Coefficients. Shows which features increase or decrease Alzheimer risk according to logistic regression

ROC Curve and Confusion Matrix Reduced-Feature Experiment

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

# 1. Calculate the ROC Curve data
fpr_red, tpr_red, _ = roc_curve(y_test_red, y_prob_red)
roc_auc_red = auc(fpr_red, tpr_red)

# 2. Create a side-by-side plot
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# --- Plot A: ROC Curve ---
ax[0].plot(fpr_red, tpr_red, color='darkorange', lw=2, label=f'Reduced Model (AUC = {roc_auc_red:.2f})')
ax[0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guess (AUC = 0.50)')
ax[0].set_xlim([0.0, 1.0])
ax[0].set_ylim([0.0, 1.05])
ax[0].set_xlabel('False Positive Rate')
ax[0].set_ylabel('True Positive Rate')
ax[0].set_title('ROC Curve: Lifestyle/Demographics Only')
ax[0].legend(loc="lower right")
ax[0].grid(alpha=0.3)

# --- Plot B: Confusion Matrix ---
# Use a threshold of 0.5 for binary classification
y_pred_red = (y_prob_red > 0.5).astype(int)
cm_red = confusion_matrix(y_test_red, y_pred_red)

disp = ConfusionMatrixDisplay(confusion_matrix=cm_red, display_labels=['No AD', 'AD'])
disp.plot(ax=ax[1], cmap='Blues', colorbar=False)
ax[1].set_title('Confusion Matrix: Reduced Model')

plt.tight_layout()
plt.show()

"Interpretability" visualization

In [ ]:
import matplotlib.pyplot as plt

# 1. Use .iloc[1:] to skip the first row (the intercept)
# This avoids KeyError if the name is 'const' vs '(Intercept)'
plot_table = red_summary_table.iloc[1:]

plt.figure(figsize=(10, 8))

# 2. Create the Forest Plot
plt.errorbar(plot_table['Odds Ratio (OR)'],
             plot_table.index,
             xerr=[plot_table['Odds Ratio (OR)'] - plot_table['95% CI Lower'],
                   plot_table['95% CI Upper'] - plot_table['Odds Ratio (OR)']],
             fmt='o', color='black', capsize=5, markersize=8)

# 3. Add the "Line of No Effect" at OR = 1
plt.axvline(x=1, color='red', linestyle='--', linewidth=1.5, label='No Effect (OR=1)')

# 4. Final Formatting
plt.xlabel('Odds Ratio (with 95% Confidence Interval)', fontsize=12)
plt.ylabel('Clinical Features', fontsize=12)
plt.title('Interpretability: Impact of Risk Factors (Reduced Model)', fontsize=14)
plt.grid(axis='x', linestyle=':', alpha=0.6)

# Use a log scale for the X-axis - this is standard for Odds Ratios
# because it makes the distance for OR=0.5 equal to OR=2.0
plt.xscale('log')

# Clean up the X-axis ticks so they are readable
from matplotlib.ticker import ScalarFormatter
plt.gca().xaxis.set_major_formatter(ScalarFormatter())
plt.xticks([0.1, 0.5, 1, 2, 5, 10])

plt.legend()
plt.tight_layout()

plt.show()

5-Fold Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Logistic Regression CV
lr_cv_scores = cross_val_score(log_reg, X_train_scaled, y_train, cv=cv, scoring='roc_auc')
print("Logistic Regression 5-Fold ROC-AUC:", lr_cv_scores.mean(), "+/-", lr_cv_scores.std())


Feature Significance / p-values

In [ ]:
import statsmodels.api as sm

# Convert to numeric arrays
y_train_numeric = pd.to_numeric(y_train, errors='coerce').values.astype(float)
X_train_sm = sm.add_constant(X_train.values.astype(float))

# Check for NaN values
print(f"NaN values in y_train: {np.isnan(y_train_numeric).sum()}")
print(f"NaN values in X_train: {np.isnan(X_train_sm).sum()}")
print(f"y_train unique values: {np.unique(y_train_numeric)}")

model_sm = sm.Logit(y_train_numeric, X_train_sm)
result = model_sm.fit(disp=0)
feature_names = ['const'] + list(X_train.columns)
print(result.summary(xname=feature_names))

Odd-ratio table

In [ ]:
# 1. Get coefficients and confidence intervals
params = result.params
conf = result.conf_int()

# 2. Convert to DataFrame immediately to avoid the IndexError
or_table = pd.DataFrame(conf, columns=['Lower 95%', 'Upper 95%'])

# 3. Add the other metrics
or_table['OR'] = params
or_table['p-value'] = result.pvalues

# 4. Exponentiate the columns (except p-value) to get actual Odds Ratios
# We only exponentiate the first three columns
or_table[['Lower 95%', 'Upper 95%', 'OR']] = np.exp(or_table[['Lower 95%', 'Upper 95%', 'OR']])

# 5. Map back the feature names so it's readable for your thesis
# 'const' is the first, followed by your X columns
feature_names = ['const'] + list(X_train.columns)
or_table.index = feature_names

print("--- Odds Ratio Table for Thesis ---")
display(or_table.round(4))

Unified Results Table

In [ ]:
import pandas as pd
import numpy as np

# 1. Extract results (these are already numpy arrays in your Colab environment)
params = result.params
std_err = result.bse
p_values = result.pvalues
conf = result.conf_int()

# 2. Build the DataFrame using the arrays directly
summary_table = pd.DataFrame(index=range(len(params)))

summary_table['Coefficient (B)'] = params
summary_table['Std. Error'] = std_err
summary_table['P-Value'] = p_values
summary_table['Odds Ratio (OR)'] = np.exp(params)

# 3. Extract Lower and Upper CI bounds (conf is a 2D array)
summary_table['95% CI Lower'] = np.exp(conf[:, 0])
summary_table['95% CI Upper'] = np.exp(conf[:, 1])

# 4. Add Significance Stars
def get_stars(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else: return ''

summary_table['Sig.'] = summary_table['P-Value'].apply(get_stars)

# 5. Map the Feature Names
# Ensure X_train is your DataFrame from the REDUCED model
feature_names = ['(Intercept)'] + list(X_train.columns)
summary_table.index = feature_names

# 6. Final Formatting and Reordering
final_cols = ['Coefficient (B)', 'Std. Error', 'P-Value', 'Sig.', 'Odds Ratio (OR)', '95% CI Lower', '95% CI Upper']
summary_table = summary_table[final_cols]

print("--- Final Model Results Table ---")
display(summary_table.round(4))

In [ ]:
plt.figure(figsize=(8, 6))

# Plot Full Model (Replace 'fpr_full' and 'tpr_full' with your baseline variables)
plt.plot(fpr_full, tpr_full, color='blue', label=f'Full Baseline (AUC = {roc_auc_full:.2f})')

# Plot Reduced Model
plt.plot(fpr_red, tpr_red, color='red', linestyle='--', label=f'Reduced Lifestyle (AUC = {roc_auc_red:.2f})')

# Plot Random Line
plt.plot([0, 1], [0, 1], color='black', linestyle=':', label='Random Guessing')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Comparison: Clinical Markers vs. Lifestyle Factors')
plt.legend()
plt.grid(alpha=0.2)
plt.show()